# Orion-XAI — QLoRA Fine-Tuning Notebook
**Model:** Llama-3.2-3B-Instruct  
**Method:** QLoRA (4-bit NF4) via Unsloth  
**Dataset:** xai_cybersec_1000.jsonl (1000 XAI/cybersecurity Alpaca examples)  
**Hardware:** Kaggle T4 GPU (16 GB VRAM)  

> After training, download the saved adapter from `/kaggle/working/orion-xai-adapter/`

## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install unsloth trl peft bitsandbytes accelerate datasets transformers --upgrade -q

## Step 2 — Load base model with 4-bit quantization

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None           # Auto-detect (float16 on T4)
LOAD_IN_4BIT = True    # QLoRA 4-bit NF4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name  = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

print("Base model loaded.")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Step 3 — Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,          # LoRA rank
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha     = 32,          # alpha = 2 * r
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth optimised checkpointing
    random_state   = 42,
    use_rslora     = False,
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## Step 4 — Load and format the dataset

In [ ]:
from datasets import load_dataset, concatenate_datasets
import os, glob

# Find all JSONL files from attached Kaggle dataset
all_files = sorted(glob.glob('/kaggle/input/**/*.jsonl', recursive=True))
print(f'Found {len(all_files)} JSONL files:')
for f in all_files:
    print(f'  {f}')

assert len(all_files) > 0, 'No JSONL files found! Attach your dataset under Input.'

datasets_list = []
for f in all_files:
    try:
        ds = load_dataset('json', data_files=f, split='train')
        print(f'  Loaded {len(ds)} examples from {os.path.basename(f)}')
        datasets_list.append(ds)
    except Exception as e:
        print(f'  SKIP {f}: {e}')

raw_dataset = concatenate_datasets(datasets_list)
print(f'\nTotal examples loaded: {len(raw_dataset)}')
print('Sample:', raw_dataset[0])


In [ ]:
# Alpaca prompt template
ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token  # <|eot_id|> for Llama 3

def format_example(examples):
    texts = []
    for instr, inp, out in zip(
        examples["instruction"],
        examples["input"],
        examples["output"],
    ):
        text = ALPACA_PROMPT.format(
            instruction=instr,
            input=inp if inp else "",
            output=out,
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = raw_dataset.map(format_example, batched=True)

# Quick sanity check — print one formatted example
print(dataset[0]["text"][:500])

## Step 5 — Configure training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = dataset,
    dataset_text_field = 'text',
    max_seq_length   = 1024,          # Reduced from 2048 to fit T4 VRAM
    dataset_num_proc = 2,
    packing          = False,          # Disabled — packing doubles effective batch, causes OOM
    args = TrainingArguments(
        per_device_train_batch_size   = 2,
        gradient_accumulation_steps   = 8,   # Effective batch = 16 (same as before)
        warmup_ratio                  = 0.05,
        num_train_epochs              = 3,
        learning_rate                 = 2e-4,
        fp16                          = not is_bfloat16_supported(),
        bf16                          = is_bfloat16_supported(),
        logging_steps                 = 10,
        optim                         = 'adamw_8bit',
        weight_decay                  = 0.01,
        lr_scheduler_type             = 'cosine',
        seed                          = 42,
        output_dir                    = '/kaggle/working/orion-xai-checkpoints',
        save_strategy                 = 'epoch',
        report_to                     = 'none',
    ),
)

print('Trainer configured.')


## Step 6 — Train

In [ ]:
# ── Confirmed fix: patch transformers trainer.py before training ─────────────
# This converts int num_items_in_batch → None before Unsloth's compiled step
# sees it. Confirmed working in previous successful run.
import shutil, os, torch

# 1. Patch transformers trainer.py
trainer_path = '/usr/local/lib/python3.12/dist-packages/transformers/trainer.py'
with open(trainer_path, 'r') as f:
    src = f.read()

old = 'tr_loss_step = self.training_step(model, inputs, num_items_in_batch)'
new = 'tr_loss_step = self.training_step(model, inputs, None if isinstance(num_items_in_batch, int) else num_items_in_batch)'

if old in src:
    with open(trainer_path, 'w') as f:
        f.write(src.replace(old, new))
    print('✅ trainer.py patched')
else:
    print('Already patched or pattern changed')

# 2. Clear compiled cache
cache = '/kaggle/working/unsloth_compiled_cache'
if os.path.exists(cache):
    shutil.rmtree(cache)
    print('✅ Compiled cache cleared')

# 3. Reset Dynamo
torch._dynamo.reset()
print('✅ Dynamo reset — ready to train')


In [ ]:
# Show GPU stats before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
max_memory = round(gpu_stats.total_memory / 1024**3, 2)
print(f"GPU: {gpu_stats.name} | VRAM: {max_memory} GB | Reserved: {start_gpu_memory} GB")

# Train
trainer_stats = trainer.train()

# Show GPU stats after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"\nTraining complete!")
print(f"Peak GPU memory: {used_memory} GB / {max_memory} GB")
print(f"Total training time: {trainer_stats.metrics['train_runtime']:.0f}s ({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"Train loss: {trainer_stats.metrics['train_loss']:.4f}")

## Step 7 — Test the model before saving

In [ ]:
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

test_prompt = ALPACA_PROMPT.format(
    instruction="Explain what SHAP values are and why they are used in cybersecurity ML models.",
    input="",
    output="",  # Empty — model fills this in
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens   = 400,
    temperature      = 0.7,
    do_sample        = True,
    pad_token_id     = tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=" * 60)
print("ORION-XAI RESPONSE:")
print("=" * 60)
print(response.strip())

## Step 8 — Save LoRA adapter

In [ ]:
ADAPTER_PATH = "/kaggle/working/orion-xai-adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print(f"Adapter saved to: {ADAPTER_PATH}")
print("\nFiles saved:")
import os
for f in os.listdir(ADAPTER_PATH):
    size = os.path.getsize(os.path.join(ADAPTER_PATH, f)) / 1e6
    print(f"  {f}  ({size:.1f} MB)")

In [ ]:
## Step 8b — Upload adapter to HuggingFace Hub
# Run this immediately after training to preserve the adapter
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, create_repo

try:
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')  # Set in Add-ons > Secrets
except:
    HF_TOKEN = ''  # Paste token here if secret not set

HF_USERNAME = 'SnowIbrahim'  # your HuggingFace username

api = HfApi()
repo_id = HF_USERNAME + '/orion-xai-adapter'

try:
    create_repo(repo_id, token=HF_TOKEN, repo_type='model', exist_ok=True)
    print('Repo ready: ' + repo_id)
except Exception as e:
    print('Repo: ' + str(e))

try:
    api.upload_folder(
        folder_path='/kaggle/working/orion-xai-adapter',
        repo_id=repo_id,
        repo_type='model',
        token=HF_TOKEN
    )
    print('Done! https://huggingface.co/' + repo_id)
except Exception as e:
    print('Upload error: ' + str(e))


## Step 9 — (Optional) Merge + save full model

Only run this if you want to save the merged model directly from Kaggle.
The merged model is ~6 GB — may hit Kaggle's 20 GB output limit.
**Recommended:** download the adapter (~50 MB) and merge locally.


In [ ]:
# OPTIONAL — run only if you want merged model on Kaggle
# Uncomment to use:

# MERGED_PATH = "/kaggle/working/orion-xai-merged"
# model.save_pretrained_merged(
#     MERGED_PATH,
#     tokenizer,
#     save_method = "merged_16bit",
# )
# print(f"Merged model saved to: {MERGED_PATH}")

print("Skipping merge — download adapter and merge locally.")
print("Download: /kaggle/working/orion-xai-adapter/")

## Done! ✅

**Next steps after downloading the adapter:**

```bash
# 1. Merge adapter with base model (run on your Windows machine)
python merge_and_convert.py

# 2. Convert merged model to GGUF
python llama.cpp/convert_hf_to_gguf.py orion-xai-merged --outtype f16 --outfile orion-xai-f16.gguf

# 3. Reload into Ollama
ollama rm orion-xai
ollama create orion-xai -f Modelfile

# 4. Test
ollama run orion-xai "Explain SHAP values for IDS detection"
```